In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Unified LLM Evaluation Suite - MBPP 提速版(方案B: 关闭Padding 最稳)
无填充、等长批次，分数与原版单条完全一致，保留批量加速
日志文件: T2.log
"""

import os, csv, json, re, torch, urllib.request, logging
from tqdm import tqdm
from modelscope import AutoModelForCausalLM, AutoTokenizer
from collections import defaultdict

# ============================ 全局推理加速 ============================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

# ============================ 日志配置（原版原样） ============================
LOG_FILE = 'T2.log'
logger = logging.getLogger('EvalSuite')
logger.setLevel(logging.INFO)

file_handler = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
file_handler.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
file_handler.setFormatter(formatter)
console_handler.setFormatter(logging.Formatter('%(message)s'))

logger.addHandler(file_handler)
logger.addHandler(console_handler)

def log(msg):
    logger.info(msg)

# ============================ 配置区 ============================
TEST_MODE = False
MAX_TEST = 5
OUTPUT_CSV = 'sore.csv'

# 批次大小，根据显存微调
BATCH_SIZE = 16

MODEL_CONFIGS = [
    {"name": "best_model_gen_05",      "path": "./best_model_gen_05"},
    {"name": "Qwen3-4B-Instruct-2507", "path": "unsloth/Qwen3-4B-Instruct-2507"},
]

DATASET_URLS = {
    "MBPP": ("mbpp.jsonl", "https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl"),
}

# ============================ 下载函数（原版原样） ============================
def ensure_downloaded(name, local_path, url):
    if os.path.exists(local_path):
        log(f"✅ {name}: 本地已存在 {local_path}")
        return True
    log(f"⬇️ {name}: 正在下载 {url}")
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=60) as response:
            with open(local_path, 'wb') as f:
                f.write(response.read())
        size = os.path.getsize(local_path)
        log(f"✅ {name}: 下载成功 ({size:,} bytes)")
        return True
    except Exception as e:
        log(f"❌ {name}: 下载失败: {e}")
        log(f"   💡 请手动下载: {url}")
        log(f"   💡 保存到: {os.path.abspath(local_path)}")
        return False

# ============================ 工具函数（原版原样） ============================
def extract_code(text):
    code = re.findall(r'```python\n(.*?)\n```', text, re.DOTALL)
    if code:
        return code[-1]
    code = re.findall(r'```\n(.*?)\n```', text, re.DOTALL)
    if code:
        return code[-1]
    return text.strip()

# ============================ 数据加载器（原版原样） ============================
def load_mbpp():
    log("【MBPP】开始加载数据...")
    local, url = DATASET_URLS["MBPP"]
    if not ensure_downloaded("MBPP", local, url):
        log("⚠️ MBPP: 数据加载失败，返回空列表")
        return []
    data = []
    with open(local, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    item = json.loads(line)
                    data.append({
                        'task_id': item.get('task_id', ''),
                        'text': item.get('text', ''),
                        'code': item.get('code', ''),
                        'test_list': item.get('test_list', [])
                    })
                except:
                    continue
    log(f"✅ MBPP: 加载完成，共 {len(data)} 条标准数据")
    return data

# ============================ 评估函数【方案B：关闭Padding + 等长分组】 ============================
def eval_mbpp(model, tokenizer, data, batch_size):
    log(f"【MBPP】开始评测，共 {len(data)} 条...")
    correct = 0
    total = len(data)
    device = next(model.parameters()).device

    # 生成参数完全对齐原版
    gen_kwargs = {
        "max_new_tokens": 512,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id,
        "use_cache": True
    }

    # 步骤1：预拼接prompt + 按token长度分组（保证组内等长，才能关闭padding）
    prompt_list = []
    for item in data:
        prompt = item['text'] + "\nWrite a function to solve this problem."
        prompt_list.append(prompt)

    # 按token长度分组，key=token长度，value=该长度下所有(索引, prompt)
    len_group = defaultdict(list)
    for idx, p in enumerate(prompt_list):
        token_len = len(tokenizer.encode(p, truncation=True, max_length=1024))
        len_group[token_len].append(idx)

    processed_count = 0

    # 步骤2：遍历每一个等长分组，组内批量推理（padding=False）
    for token_len, idx_list in len_group.items():
        group_size = len(idx_list)
        # 修复语法错误：for start in 而非 for start =
        for start in range(0, group_size, batch_size):
            end = min(start + batch_size, group_size)
            batch_idx = idx_list[start:end]
            batch_prompts = [prompt_list[i] for i in batch_idx]

            # 关键：padding=False，无任何填充，和单条输入格式一致
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
                padding=False
            ).to(device)

            # 批量生成
            with torch.no_grad():
                outputs = model.generate(**inputs, **gen_kwargs)

            # 逐条解码、判分（完全复刻原版逻辑）
            for i in range(len(batch_prompts)):
                gen_text = tokenizer.decode(
                    outputs[i][inputs["input_ids"][i].shape[0]:],
                    skip_special_tokens=True
                ).strip()
                code = extract_code(gen_text)
                if 'def ' in code and len(code) > 30:
                    correct += 1

            processed_count += len(batch_idx)

            # 原版进度打印规则
            if processed_count % 50 == 0 or processed_count == total:
                log(f"  MBPP 进度: {processed_count}/{total} ({processed_count/total*100:.1f}%) | 正确: {correct}/{processed_count} ({correct/processed_count*100:.2f}%)")

            # 释放显存
            del inputs, outputs
            torch.cuda.empty_cache()

    acc = correct / total * 100
    log(f"✅ MBPP 评测完成: {acc:.2f}% ({correct}/{total})")
    return acc

# ============================ 主程序（原版结构不变） ============================
EVAL = {
    "MBPP": (load_mbpp, eval_mbpp),
}

def main():
    log("=" * 70)
    log(f"Unified LLM Evaluation Suite | {'TEST MODE (5 samples each)' if TEST_MODE else 'FULL MODE'}")
    log("=" * 70)
    log("【阶段1】加载数据集...")
    log("-" * 70)

    datasets = {}
    for name, (loader, _) in EVAL.items():
        try:
            d = loader()
            datasets[name] = d[:MAX_TEST] if TEST_MODE else d
            log(f"✅ {name:<25} {len(datasets[name]):>5} samples")
        except Exception as e:
            log(f"❌ {name:<25} ERROR: {e}")
            datasets[name] = []

    log("-" * 70)
    log("【阶段2】模型评测...")
    log("-" * 70)

    results = []
    for cfg in MODEL_CONFIGS:
        log(f"\n>>> 开始评估模型: {cfg['name']}")
        log(f"   加载模型: {cfg['path']}")

        tok = AutoTokenizer.from_pretrained(cfg['path'], trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            cfg['path'],
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        model.eval()
        log(f"   ✅ 模型加载完成")

        for name, (_, ev) in EVAL.items():
            d = datasets.get(name, [])
            if not d:
                log(f"⚠️ {name}: 无数据，跳过")
                results.append({"model": cfg['name'], "dataset": name, "accuracy": 0.0, "samples": 0})
                continue

            log(f"\n   >>> 开始评测 {name} ({len(d)} samples)")
            try:
                acc = ev(model, tok, d, BATCH_SIZE)
                results.append({
                    "model": cfg['name'],
                    "dataset": name,
                    "accuracy": round(acc, 2),
                    "samples": len(d)
                })
            except Exception as e:
                log(f"   ❌ {name} 评测失败: {str(e)[:100]}")
                results.append({"model": cfg['name'], "dataset": name, "accuracy": 0.0, "samples": 0})

        del model
        torch.cuda.empty_cache()
        log(f"✅ 模型 {cfg['name']} 评测完成，显存已释放")

    # 输出CSV
    log("\n【阶段3】保存结果...")
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["model", "dataset", "accuracy", "samples"])
        writer.writeheader()
        writer.writerows(results)
    log(f"✅ 结果已保存到: {os.path.abspath(OUTPUT_CSV)}")

    # 汇总打印
    log("\n" + "=" * 70)
    log("📊 评测结果汇总")
    log("=" * 70)
    for cfg in MODEL_CONFIGS:
        log(f"\n【{cfg['name']}】")
        for r in results:
            if r['model'] == cfg['name']:
                log(f"  {r['dataset']:<25} {r['accuracy']:>6.2f}%  ({r['samples']} samples)")

    log("\n" + "=" * 70)
    log(f"📁 日志文件: {os.path.abspath(LOG_FILE)}")
    log(f"📁 结果文件: {os.path.abspath(OUTPUT_CSV)}")
    log("=" * 70)
    log("✅ 全部评测完成")

if __name__ == "__main__":
    main()

Unified LLM Evaluation Suite | FULL MODE
Unified LLM Evaluation Suite | FULL MODE
【阶段1】加载数据集...
【阶段1】加载数据集...
----------------------------------------------------------------------
----------------------------------------------------------------------
【MBPP】开始加载数据...
【MBPP】开始加载数据...
✅ MBPP: 本地已存在 mbpp.jsonl
✅ MBPP: 本地已存在 mbpp.jsonl
✅ MBPP: 加载完成，共 974 条标准数据
✅ MBPP: 加载完成，共 974 条标准数据
✅ MBPP                        974 samples
✅ MBPP                        974 samples
----------------------------------------------------------------------
----------------------------------------------------------------------
【阶段2】模型评测...
【阶段2】模型评测...
----------------------------------------------------------------------
----------------------------------------------------------------------

>>> 开始评估模型: best_model_gen_05

>>> 开始评估模型: best_model_gen_05
   加载模型: ./best_model_gen_05
   加载模型: ./best_model_gen_05


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ 模型加载完成
   ✅ 模型加载完成

   >>> 开始评测 MBPP (974 samples)

   >>> 开始评测 MBPP (974 samples)
【MBPP】开始评测，共 974 条...
【MBPP】开始评测，共 974 条...
  MBPP 进度: 974/974 (100.0%) | 正确: 835/974 (85.73%)
  MBPP 进度: 974/974 (100.0%) | 正确: 835/974 (85.73%)
✅ MBPP 评测完成: 85.73% (835/974)
✅ MBPP 评测完成: 85.73% (835/974)
✅ 模型 best_model_gen_05 评测完成，显存已释放
✅ 模型 best_model_gen_05 评测完成，显存已释放

>>> 开始评估模型: Qwen3-4B-Instruct-2507

>>> 开始评估模型: Qwen3-4B-Instruct-2507
   加载模型: unsloth/Qwen3-4B-Instruct-2507
   加载模型: unsloth/Qwen3-4B-Instruct-2507


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ 模型加载完成
   ✅ 模型加载完成

   >>> 开始评测 MBPP (974 samples)

   >>> 开始评测 MBPP (974 samples)
【MBPP】开始评测，共 974 条...
【MBPP】开始评测，共 974 条...
  MBPP 进度: 974/974 (100.0%) | 正确: 730/974 (74.95%)
  MBPP 进度: 974/974 (100.0%) | 正确: 730/974 (74.95%)
✅ MBPP 评测完成: 74.95% (730/974)
✅ MBPP 评测完成: 74.95% (730/974)
✅ 模型 Qwen3-4B-Instruct-2507 评测完成，显存已释放
✅ 模型 Qwen3-4B-Instruct-2507 评测完成，显存已释放

【阶段3】保存结果...

【阶段3】保存结果...
✅ 结果已保存到: /root/autodl-tmp/sore.csv
✅ 结果已保存到: /root/autodl-tmp/sore.csv


📊 评测结果汇总
📊 评测结果汇总

【best_model_gen_05】

【best_model_gen_05】
  MBPP                       85.73%  (974 samples)
  MBPP                       85.73%  (974 samples)

【Qwen3-4B-Instruct-2507】

【Qwen3-4B-Instruct-2507】
  MBPP                       74.95%  (974 samples)
  MBPP                       74.95%  (974 samples)


📁 日志文件: /root/autodl-tmp/T2.log
📁 日志文件: /root/autodl-tmp/T2.log
📁 结果文件: /root/autodl-tmp/sore.csv
📁 结果文件: /root/autodl-tmp/sore.csv
✅ 全部评测完成
✅ 全部评测完成
